# 6.4 Multi-Drone Search with LangGraph

# Tutorial: Building a Multi-Agent System

In this section, we build a multi-agent system step by step.

## 1. Define Graph State

The graph state is the "Single Source of Truth" - a shared object that all nodes can read from and write to.

We use Python's `TypedDict` to define the state fields and their types, which LangGraph uses for validation and management.

In [ ]:
from typing import TypedDict, Annotated, List, Dict
from langchain_core.messages import BaseMessage
import operator

class GraphState(TypedDict):
    """
    Graph state definition for the multi-agent drone system.
    """
    mission_goal: str

    # task_queue: no Annotated (replaced entirely each time)
    task_queue: List[Dict] 
    # completed_tasks: uses Annotated (appends new tasks)
    completed_tasks: Annotated[List[Dict], operator.add]

    # messages: append-only message history
    messages: Annotated[List[BaseMessage], operator.add]

## 2. Build the Supervisor Node

The supervisor node is a Python function that receives the current GraphState as input and uses the LLM to decompose tasks.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage

# Using io.solutions model (OpenAI-compatible interface)
llm = ChatOpenAI(
    base_url="https://api.intelligence.io.solutions/api/v1",
    api_key="80e68c38-22cb-4f71-9377-0768c4d7fe15",
    model="moonshotai/Kimi-K2.6",  # Replace with your model ID
    temperature=0.01,
)

def supervisor_node(state: GraphState):
    """
    Supervisor node: decomposes the mission into tasks and assigns them.
    """
    print("--- Supervisor Node ---")

    # If task queue is empty, this is the first run - need to decompose the task
    if not state['task_queue']:
        system_prompt = (
            "You are a drone mission planner. Your task is to receive a high-level objective "
            "and decompose it into sub-tasks for individual drones (Drone1, Drone2). "
            "Each task should be a dictionary containing 'drone_id' and 'target' (a list of [x, y, z] coordinates). "
            "Output the tasks as a JSON list only, with no other text. "
            "Z represents altitude: e.g., -10 means 10 meters above ground."
        )

        prompt = ChatPromptTemplate.from_messages([
            ("system", system_prompt),
            ("human", "Mission objective: {mission_goal}")
        ])

        chain = prompt | llm
        response = chain.invoke({"mission_goal": state['mission_goal']})

        import json
        try:
            tasks = json.loads(response.content)
            # Verify it's a list
            if isinstance(tasks, list):
                print(f"Decomposed into tasks: {tasks}")
            else:
                tasks = []  # fallback
                print("Warning: LLM output was not a list, using empty task queue")
        except json.JSONDecodeError:
            tasks = []  # fallback
            print("Warning: LLM output was not valid JSON, using empty task queue")

        # Update state: return new task list
        return {"task_queue": tasks, "messages": [HumanMessage(content=f"Tasks assigned: {tasks}")]}
    else:
        # If task queue is not empty, no operation needed
        return {}

The LLM is given a clear system prompt specifying the expected input and output format (JSON). This helps the model produce structured, parseable output.

## 3. Build the Scout Node

The scout node controls individual drones. It takes tasks from the queue and executes them.

In [ ]:
from langchain_core.messages import AIMessage, ToolMessage
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from airsim_agent import *  # takeoff, fly_to_position

# Create tool node for executing tool calls
tool_node = ToolNode([takeoff, fly_to_position])

def scout_node(state: GraphState):
    """
    Scout node: executes drone flight tasks.
    """
    print("--- Scout Node ---")

    if state['task_queue']:
        task = state['task_queue'][0]  # Get first task from queue
        drone_id = task['drone_id']
        target = task['target']  # [x, y, z]

        # Prompt with formatted parameters
        system_prompt = (
            "You are a drone controller. You MUST call tools in the following order to complete the task: "
            "1. Call takeoff(drone_id='{drone_id}'). "
            "2. Then call fly_to_position(drone_id='{drone_id}', x={x}, y={y}, z={z}) to fly to position. "
            "Do not output any other text, only output tool calls. Parameters must be exact."
        ).format(drone_id=drone_id, x=target[0], y=target[1], z=target[2])

        human_input = f"Execute now: {drone_id} fly to {target}."

        prompt = ChatPromptTemplate.from_messages([
            ("system", system_prompt),
            ("human", human_input)
        ])

        llm_with_tools = llm.bind_tools([takeoff, fly_to_position])
        chain = prompt | llm_with_tools
        ai_message = chain.invoke({})

        # Debug: print LLM output
        print(f"LLM output for {drone_id}: {ai_message.content[:100]}...")
        if hasattr(ai_message, 'tool_calls'):
            print(f"Tool calls: {len(ai_message.tool_calls)}")

        messages_to_add = [ai_message]

        # Fallback: if no tool calls generated, execute manually
        if not hasattr(ai_message, 'tool_calls') or not ai_message.tool_calls:
            print(f"Warning: {drone_id} did not generate tool calls, executing manually.")
            # Takeoff
            takeoff_result = takeoff.invoke({"drone_id": drone_id})
            messages_to_add.append(ToolMessage(content=takeoff_result, tool_call_id="manual_takeoff"))
            # Fly to position
            fly_result = fly_to_position.invoke({"drone_id": drone_id, "x": target[0], "y": target[1], "z": target[2]})
            messages_to_add.append(ToolMessage(content=fly_result, tool_call_id="manual_fly"))
            # Create simulated AI message for router
            ai_message = AIMessage(content=f"{drone_id} executed successfully.")
            messages_to_add = [ai_message] + messages_to_add[1:]

        # Update state: remove completed task from queue
        remaining_tasks = state['task_queue'][1:]  # Remove first task
        new_completed = [task]  # Add to completed (Annotated will append)

        return {
            "task_queue": remaining_tasks,  # Replace queue (no Annotated)
            "completed_tasks": new_completed,  # Append to completed tasks
            "messages": messages_to_add
        }
    else:
        print("All tasks completed!")
        return {"messages": [HumanMessage(content="All tasks completed")]}

## 4. Assemble the Graph

Now we use LangGraph's StateGraph to connect all the nodes together.

In [ ]:
# Create graph
workflow = StateGraph(GraphState)

# Add nodes
workflow.add_node("supervisor", supervisor_node)
workflow.add_node("scout", scout_node)
workflow.add_node("tool_executor", tool_node)

# Set entry point
workflow.set_entry_point("supervisor")


# Router function: determines next step based on state
def router(state: GraphState):
    print("--- Router ---")

    if not state['messages']:
        print("No messages, ending")
        return END

    last_message = state['messages'][-1]

    if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
        print("Tool calls detected, routing to tool_executor")
        return "tool_executor"

    if state['task_queue']:
        print(f"Still have {len(state['task_queue'])} tasks, continuing to scout")
        return "scout"

    print("All tasks complete, ending")
    return END

# Define edges
workflow.add_edge("supervisor", "scout")
workflow.add_conditional_edges(
    "scout", 
    router, 
    {"tool_executor": "tool_executor", "scout": "scout", END: END}
)
workflow.add_edge("tool_executor", "scout")

# Compile the graph
app = workflow.compile()

The router function is the key control mechanism. It checks the current graph state and determines which node to execute next. This is what makes the graph dynamic.

## 5. Execute and Observe

With the graph compiled, we provide an initial state and run the multi-agent system.

In [ ]:
# Run with max_iterations limit
initial_state = {
    "mission_goal": "Send Drone1 to position (-10, 10, -10) and Drone2 to position (10, 10, -10).",
    "task_queue": [],
    "completed_tasks": [],
    "messages": []
}

for event in app.stream(initial_state, {"recursion_limit": 100, "max_iterations": 20}):
    for key, value in event.items():
        print(f"Node: {key}")
        print("---")
        print(value)
        print("\n===================\n")

# Check final state
final_state = app.invoke(initial_state, config={"max_iterations": 20})
print("Final state summary:")
print(f"Remaining tasks: {len(final_state['task_queue'])}")
print(f"Completed tasks: {final_state['completed_tasks']}")
print(f"Messages count: {len(final_state['messages'])}")

### Result Analysis

The drone tasks were executed successfully. From the logs, the run demonstrates that the multi-agent system works correctly.

#### 1. Process Confirmation

- **First run (stream)**:
  - **Supervisor**: The LLM decomposes the objective into 2 tasks (Drone1 and Drone2), populating `task_queue` successfully.
  - **Scout (Drone1)**: If the LLM generates tool calls, they are executed via tool_executor. If not (Tool calls: 0), the fallback mechanism triggers manual execution:
    - Calls `takeoff("Drone1")` - drone takes off in AirSim.
    - Calls `fly_to_position("Drone1", -10, 10, -10)` - drone flies to target.
    - Updates state: `task_queue` = [Drone2 task], `completed_tasks` = [Drone1 task].
  - **Router**: Tasks remain -> routes to "scout" (continue).
  - **Scout (Drone2)**: Same process, executes successfully. `task_queue: []`, `completed_tasks: [Drone1, Drone2]`.
  - **Router**: No tasks remaining -> END.

- **Second run (invoke)**:
  - `app.invoke(initial_state)` creates a fresh instance from the initial state (`task_queue: []`), so the LLM decomposes tasks again.
  - Both drones execute their tasks. Final state shows 0 remaining tasks, 2 completed tasks.

**Conclusion**: Both runs successfully achieve the objective:
- Drones take off and fly to targets in AirSim.
- State updates correctly (queue empties, completed list fills).
- The fallback mechanism provides robustness even when the LLM doesn't generate tool calls.
- No KeyError or TypeError errors - the graph/edge definitions are correct.

#### 2. Notes and Observations
- **Duplicate execution**: `app.stream()` and `app.invoke()` both execute from `initial_state`. For production use, run only one.
- **LLM tool calling reliability**: The model may not always generate tool calls reliably. The fallback mechanism ensures tasks complete regardless.
- **Message accumulation**: Messages accumulate via the Annotated[add] pattern, providing a full execution log.

# Execution and Observation

When running, watch the AirSim window. `Drone1` and `Drone2` will take off sequentially (or in parallel if async tools are used) and fly to their target positions.

The console will print each node's input and output, showing the full pipeline:
1. **Task decomposition**
2. **Tool call requests**
3. **Tool execution results**

## Implementation: ReAct Pattern

This system implements the ReAct pattern where agents can:

- **Reason**: The supervisor node uses the LLM to plan, decompose, and decide.
- **Act**: AirSim API tool functions execute real actions and return observations.

This is a concrete **implementation** of the **ReAct (Reason + Act)** paradigm for embodied agents.

> Reference: [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629)

## Value

Through this project, we're not just "telling a drone to fly" - we're implementing an **agent reasoning-action loop** in a high-fidelity 3D simulation environment.

This has significant educational and practical value:

- Real-world grounding: LLM decisions drive real (simulated) physical actions.
- Extensibility: Multi-agent coordination, state management, and more can be added.
- Transferability: The same patterns apply to robotics, autonomous vehicles, etc.

> Summary: This is not just a technology demo - it's "Learning by Doing", connecting AI concepts to real-world applications.

# Future Directions

This introduction covers the fundamentals of multi-drone agent systems. Moving from simulation to real-world deployment involves additional challenges.

## Sim-to-Real Transfer

Algorithms that work in AirSim may not directly transfer to real hardware. Real sensors have noise, latency, and limitations that don't exist in simulation. The "sim-to-real gap" is a long-standing challenge in robotics.

## LLM Challenges in Robotics

### Grounding
How do you connect LLM "understanding" to real drone sensor data? Key approaches include using sensor data to verify LLM decisions, or using it as state input for more informed reasoning.

### Safety and Reliability
If an agent generates an unsafe instruction, the consequences are real. Safety mechanisms must include:
- Pre-execution validation of LLM-generated instructions;
- Safety boundary constraints (geofencing, altitude limits);
- Human-in-the-loop oversight, allowing operators to intervene at any time.

## Cloud LLM vs. Edge LLM

This is a key architectural decision:

- **Cloud LLM**: Leverages large, powerful language models with strong reasoning capabilities, but introduces latency - problematic for drones requiring fast response times.
- **Edge LLM (on-device)**: Small models deployed directly on the drone hardware enable low latency and high autonomy, but are limited by the drone's computational resources.

A promising approach is a **cloud-edge hybrid architecture**: the cloud handles high-level planning and complex reasoning, while edge models handle real-time response and safety control, communicating via efficient protocols.

## Research Frontiers

Active research areas include:

- **Adaptive task allocation**: Using LLMs to dynamically adjust task parameters (priority, assignment, timing) based on changing conditions, such as weather or new objectives.

- **LLM + Multi-Agent Reinforcement Learning (MARL)**: Combining LLM planning and task understanding with MARL's reward-driven policy optimization, leveraging the strengths of both approaches.

These research directions point toward increasingly capable and autonomous drone systems.